<a href="https://colab.research.google.com/github/AmnaNoor123/urdu-ocr-codesaviours-si26-amna/blob/main/SI26_Week4_Amna_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This notebook covers Week 4:**

Fine-tuning TrOCR on the Urdu OCR dataset.

Up till now we collected images (Week 1), preprocessed them (Week 2), and built the dataset class + train/test split (Week 3). This week we actually train a model on that data.

TrOCR is a transformer model from Microsoft — it has a vision encoder (reads the image) and a text decoder (outputs the characters). Instead of training from scratch, we load a version already trained on printed text and fine-tune it on our Urdu images. This is called **transfer learning**, and it's why we can get decent results with a small dataset instead of needing millions of images.

In this notebook, we will:

Reload the dataset from Week 3 (repo + labels + dataset class)
Load the pretrained TrOCR model
Set up the DataLoaders and optimiser
Train the model for a few epochs
Evaluate accuracy on the test set
Save the fine-tuned model to Google Drive


Reload Week 3 Setup — Repo, Labels, Dataset Class

In [ ]:
!git clone https://github.com/AmnaNoor123/urdu-ocr-codesaviours-si26-amna.git
!cp urdu-ocr-codesaviours-si26-amna/labels.csv data/labels.csv


Cloning into 'urdu-ocr-codesaviours-si26-amna'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 77 (delta 34), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 87.75 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (34/34), done.
cp: cannot create regular file 'data/labels.csv': No such file or directory


In [ ]:
!find urdu-ocr-codesaviours-si26-amna -maxdepth 2 -type f

urdu-ocr-codesaviours-si26-amna/SI26_Week3_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/SI26_Week2_amna.ipynb
urdu-ocr-codesaviours-si26-amna/SI26_Week4_Amna_(1).ipynb
urdu-ocr-codesaviours-si26-amna/.git/index
urdu-ocr-codesaviours-si26-amna/.git/packed-refs
urdu-ocr-codesaviours-si26-amna/.git/config
urdu-ocr-codesaviours-si26-amna/.git/description
urdu-ocr-codesaviours-si26-amna/.git/HEAD
urdu-ocr-codesaviours-si26-amna/SI26_Week1_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/README.md
urdu-ocr-codesaviours-si26-amna/labels.csv


In [ ]:
!rm -rf urdu-ocr-codesaviours-si26-amna
!git clone https://github.com/AmnaNoor123/urdu-ocr-codesaviours-si26-amna.git
!find urdu-ocr-codesaviours-si26-amna -maxdepth 2 -type f

Cloning into 'urdu-ocr-codesaviours-si26-amna'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 77 (delta 34), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 87.75 KiB | 5.48 MiB/s, done.
Resolving deltas: 100% (34/34), done.
urdu-ocr-codesaviours-si26-amna/SI26_Week3_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/SI26_Week2_amna.ipynb
urdu-ocr-codesaviours-si26-amna/SI26_Week4_Amna_(1).ipynb
urdu-ocr-codesaviours-si26-amna/.git/index
urdu-ocr-codesaviours-si26-amna/.git/packed-refs
urdu-ocr-codesaviours-si26-amna/.git/config
urdu-ocr-codesaviours-si26-amna/.git/description
urdu-ocr-codesaviours-si26-amna/.git/HEAD
urdu-ocr-codesaviours-si26-amna/SI26_Week1_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/README.md
urdu-ocr-codesaviours-si26-amna/labels.csv


In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
shutil.copy('urdu-ocr-codesaviours-si26-amna/labels.csv', 'data/labels.csv')

import pandas as pd
df = pd.read_csv('data/labels.csv')
print('Total entries:', len(df))


Total entries: 200


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Books.zip', 'r') as zip_ref:
    zip_ref.extractall('data/raw')
import os
print('Unzipped folders:', os.listdir('data/raw'))


Mounted at /content/drive
Unzipped folders: ['Handwritten', 'Newspaper', 'Books', 'Other', 'Synthetic raw images', 'Sign boards']


In [ ]:
import os, shutil
import pandas as pd
import re
import unicodedata # Import unicodedata module

def clean_filename(name):
    # Ensure it's a string, strip leading/trailing whitespace first
    name = str(name).strip()

    # Convert to lowercase for case-insensitive matching (important for Linux filesystems)
    name = name.lower()

    # Normalize unicode characters to canonical composed form
    name = unicodedata.normalize('NFC', name)

    # Replace any non-standard whitespace characters with a single space
    # and strip again to handle potential internal leading/trailing spaces from this step
    name = re.sub(r'\s+', ' ', name).strip()

    base, ext = os.path.splitext(name)

    def decode_match(m):
        return chr(int(m.group(1), 16))

    # Handle #Uxxxx escapes in base name and extension
    base = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, base)
    ext = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, ext)

    # Remove trailing (number) like ' (1)', ' (2)'
    base = re.sub(r'\s*\(\d+\)$', '', base)

    full = base + ext

    # Remove duplicate extensions like '.png.png'
    full = re.sub(r'(\.\w+)\1$', r'\1', full)

    return full

# Assuming df is loaded from 'data/labels.csv'
# We need to process each entry in df['image'] and update the path.

# Step 1: Collect all actual image file paths on disk
# Maps cleaned_basename to a list of full paths where it was found
actual_image_files = {}
for root, _, files in os.walk('data/raw'):
    for fname in files:
        full_path_on_disk = os.path.join(root, fname)
        cleaned_fname_on_disk = clean_filename(fname)
        if cleaned_fname_on_disk not in actual_image_files:
            actual_image_files[cleaned_fname_on_disk] = []
        actual_image_files[cleaned_fname_on_disk].append(full_path_on_disk)

updated_image_paths = []
found_after_reconciliation_count = 0
not_found_final_count = 0
unresolved_paths_debug = [] # List to store paths that couldn't be resolved

print(f"Total unique cleaned filenames on disk: {len(actual_image_files)}")

for original_image_path_in_df in df['image']:
    dirname_from_df, basename_from_df = os.path.split(original_image_path_in_df)
    cleaned_basename_from_df = clean_filename(basename_from_df)

    resolved_path = None

    # Priority 1: Check for existence of the path derived from df with cleaned basename
    # This covers cases where original path was wrong but directory was right.
    candidate_path_from_df_cleaned_basename = os.path.join(dirname_from_df, cleaned_basename_from_df)
    if os.path.exists(candidate_path_from_df_cleaned_basename):
        resolved_path = candidate_path_from_df_cleaned_basename
    elif os.path.exists(original_image_path_in_df):
        # Fallback: if original_image_path_in_df itself exists (e.g., already clean)
        resolved_path = original_image_path_in_df

    if not resolved_path:
        # Priority 2: Deep search based on cleaned basename across all directories
        if cleaned_basename_from_df in actual_image_files:
            potential_paths_on_disk = actual_image_files[cleaned_basename_from_df]

            # Prioritize a path that's in the same directory as original_image_path_in_df
            # Compare cleaned directory names too for robustness
            cleaned_dirname_from_df = clean_filename(dirname_from_df) # Clean dirname from df entry
            matching_in_original_dir = []
            for p_disk in potential_paths_on_disk:
                disk_dirname_cleaned = clean_filename(os.path.split(p_disk)[0])
                if disk_dirname_cleaned == cleaned_dirname_from_df:
                    matching_in_original_dir.append(p_disk)

            if len(matching_in_original_dir) == 1:
                resolved_path = matching_in_original_dir[0]
                found_after_reconciliation_count += 1
            elif len(potential_paths_on_disk) == 1:
                # If unique overall, use that one (even if directory doesn't match clean_dirname_from_df)
                resolved_path = potential_paths_on_disk[0]
                found_after_reconciliation_count += 1
            elif len(potential_paths_on_disk) > 1:
                # If multiple ambiguous matches and no clear directory match, pick the first one.
                # This is a heuristic to ensure we find *an* image if multiple exist with same basename.
                resolved_path = potential_paths_on_disk[0]
                found_after_reconciliation_count += 1

    if resolved_path:
        updated_image_paths.append(resolved_path)
    else:
        not_found_final_count += 1
        updated_image_paths.append(original_image_path_in_df) # Keep original for debug, it will be filtered
        unresolved_paths_debug.append(original_image_path_in_df) # Store for debugging
        # Print debug info for each unresolved path
        print(f"DEBUG: Unresolved: {original_image_path_in_df}")
        print(f"  Cleaned basename from df: '{cleaned_basename_from_df}'")
        print(f"  Available on disk for this cleaned basename: {actual_image_files.get(cleaned_basename_from_df, 'None')}")


# Update the 'image' column in the DataFrame
df['image'] = updated_image_paths

# Save the modified DataFrame back to 'labels.csv'
df.to_csv('data/labels.csv', index=False)

print('labels.csv updated with reconciled image paths.')
print(f'Reconciled {found_after_reconciliation_count} entries via deep search and cleaning.')
if not_found_final_count > 0:
    print(f'Warning: {not_found_final_count} image paths from labels.csv were not found on disk.')
    print('These entries will be filtered by the dataset loader, resulting in fewer than 200 samples.')
    print('Unresolved paths (first 5):', unresolved_paths_debug[:5])
else:
    print('All 200 image paths from labels.csv were successfully reconciled with files on disk.')

Total unique cleaned filenames on disk: 200
labels.csv updated with reconciled image paths.
Reconciled 20 entries via deep search and cleaning.
All 200 image paths from labels.csv were successfully reconciled with files on disk.


Dataset Class (same as Week 3)

In [ ]:
!pip install transformers torch pillow pandas sentencepiece

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import os
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        initial_count = len(self.data)
        self.data['image_exists'] = self.data['image'].apply(lambda x: os.path.exists(x))
        self.data = self.data[self.data['image_exists']].drop(columns=['image_exists'])
        filtered_count = len(self.data)

        if initial_count > filtered_count:
            print(f'Warning: Removed {initial_count - filtered_count} entries due to missing image files.')
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=64,
            truncation=True
        ).input_ids
        labels = [
            label if label != self.processor.tokenizer.pad_token_id else -100
            for label in labels
        ]
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}

In [ ]:
from transformers import TrOCRProcessor, AutoImageProcessor, RobertaTokenizer

# Instantiate image processor
image_processor = AutoImageProcessor.from_pretrained('microsoft/trocr-base-printed', backend="pil")

# Instantiate tokenizer with use_fast=False
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)

# Combine them into a processor
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

dataset = UrduOCRDataset('data/labels.csv', processor)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([64])
Dataset is working correctly!
Training samples: 160
Testing samples: 40


## Step 1 — Load the Pretrained TrOCR Model


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoImageProcessor, RobertaTokenizer
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

# Load the pretrained model components
image_processor = AutoImageProcessor.from_pretrained('microsoft/trocr-base-printed', backend="pil")
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


## Step 2 — Set Up Training

A `DataLoader` wraps the dataset and feeds it to the model in small batches — batch size 4 means the model sees 4 images at a time before updating its weights. `AdamW` is the optimiser that controls how the model adjusts itself after each batch.

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# Optimiser
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 40
Ready to train!


In [ ]:
num_epochs = 12
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    processed_batches = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        try:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            processed_batches += 1 # Increment only for successful batches
            if batch_idx % 10 == 0:
                print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
        except FileNotFoundError as e:
            print(f"  Skipping batch {batch_idx} due to FileNotFoundError: {e}")
            continue # Skip to the next batch

    if processed_batches > 0:
        avg_loss = total_loss / processed_batches # Use processed_batches for accurate average
        print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')
    else:
        print(f'Epoch {epoch + 1} complete | No batches processed successfully.')

print('\nTraining complete!')


Epoch 1/12
------------------------------
  Batch 0/40 | Loss: 17.0778
  Batch 10/40 | Loss: 5.5603
  Batch 20/40 | Loss: 4.2897
  Batch 30/40 | Loss: 3.8896
Epoch 1 complete | Average Loss: 5.2877

Epoch 2/12
------------------------------
  Batch 0/40 | Loss: 3.7045
  Batch 10/40 | Loss: 3.3851
  Batch 20/40 | Loss: 3.5199
  Batch 30/40 | Loss: 3.2712
Epoch 2 complete | Average Loss: 3.5873

Epoch 3/12
------------------------------
  Batch 0/40 | Loss: 3.5268
  Batch 10/40 | Loss: 3.8069
  Batch 20/40 | Loss: 3.6970
  Batch 30/40 | Loss: 3.4515
Epoch 3 complete | Average Loss: 3.5510

Epoch 4/12
------------------------------
  Batch 0/40 | Loss: 3.8585
  Batch 10/40 | Loss: 3.5909
  Batch 20/40 | Loss: 3.4609
  Batch 30/40 | Loss: 3.6186
Epoch 4 complete | Average Loss: 3.5059

Epoch 5/12
------------------------------
  Batch 0/40 | Loss: 3.4956
  Batch 10/40 | Loss: 3.3393
  Batch 20/40 | Loss: 3.6145
  Batch 30/40 | Loss: 3.2544
Epoch 5 complete | Average Loss: 3.4829

Epoch 6/

## Step 3 — Evaluate Your Model

Now we test the model on images it has never seen — the test split from Week 3. `model.eval()` turns off weight updates so evaluation doesn't affect the trained model.

In [ ]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )
        actual_text = processor.batch_decode(
            labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')


=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: ��������������������
Actual:    ل

Predicted: 
Actual:    ظ

Predicted: اااااااااااااااااااا
Actual:    علم بڑی دولت ہے، ہم اسے حاصل کرنے کے لیے صرف محن

Predicted: ��������������������
Actual:    سچائی ہمیشہ کامیاب ہوتی ہے

Predicted: 
Actual:    ل

Predicted: 
Actual:    ۃ

Predicted: 
Actual:    ٹ

Predicted: ��������������������
Actual:    آج کا موسم خوشگوار ہے

Predicted: ��������������������
Actual:    سربراہ کے ایچ خورشید مرحوم نے آج سے نصف صدی

Predicted: ��������������������
Actual:    میں ایک سندر لڑکی ہوں

Predicted: 
Actual:    ڑ

Predicted: 
Actual:    ش

Predicted: 
Actual:    ٹ

Predicted: ��������������������
Actual:    پانی زندگی کے لیے ضروری ہے

Predicted: 
Actual:    ذ

Predicted: ��������������������
Actual:    وقت کی قدر کرنی چاہیے

Predicted: ��������������������
Actual:    میرا نام آمنہ ہے

Predicted: 
Actual:    م

Predicted: 
Actual:    ث

Predicted: 
Actual:    ط

Predicted: ��������������������
Actual:    شہر کی سڑکیں کشادہ ہیں

Predicted: 
Actual:

**Discussion Point : TrOCR Tokenizer Limitation on Urdu Script**

**Final Result:** Accuracy: 0.0% (0/40 correct) on the test set after fine-tuning for **12 epochs** on the full 200-image dataset. Loss stayed flat between 3.1 and 3.3 across all 12 epochs — it did not improve with more training.

**What went wrong:** Model predictions consistently decoded as `�` (replacement characters), blank, or repeated single characters instead of valid Urdu text.

**Why this happened:** `microsoft/trocr-base-printed` uses a RobertaTokenizer, which is a byte-level tokenizer trained on English/Latin text. Urdu characters aren't in its original vocabulary, so each one has to be reconstructed from a precise sequence of byte-level tokens — if even one byte is predicted incorrectly, the character fails to decode and shows as `�`.

**Example mismatches (from this run):**
- Actual: `علم بڑی دولت ہے، ہم اسے حاصل کرنے کے لیے صرف محن` → Predicted: `اااااااااااااااااااا` (model just repeats "ا")
- Actual: `آج کا موسم خوشگوار ہے` → Predicted: `��������������������`
- Actual: single letters (`ل`, `ظ`, `ٹ`, `ذ`, `ط`, `ع`,



In [ ]:
import os
print(os.listdir('data/raw/Books')[:5])

['ٹ_98.png', 'ژ_98.png', 'ث_984.png', 'ت_95.png', 'ط_973.png']


In [ ]:
!pip install easyocr

import easyocr
reader = easyocr.Reader(['ur'])
result = reader.readtext('data/raw/Synthetic raw images/urdu_2 (1).png', detail=0)
print(result)


['آج کاموسم خوشگوارے']


In [ ]:
result = reader.readtext('data/raw/Synthetic raw images/urdu_3 (1).png', detail=0)
print(result)

['انسا . ماحقے']


 **Exploring an Alternative — EasyOCR**

Since fine-tuning TrOCR isn't converging on Urdu within this timeline, testing EasyOCR — a ready-to-use OCR library with native Urdu ('ur') support — as a practical alternative for the Week 5 deployment.

## Step 4 — Save  Model


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model


In [ ]:
!pip install gradio easyocr

import gradio as gr
import easyocr
import numpy as np

reader = easyocr.Reader(['ur'])

def extract_urdu_text(image):
    """Takes an image, returns extracted Urdu text."""
    if image is None:
        return 'Please upload an image'
    result = reader.readtext(np.array(image), detail=0, paragraph=True)
    text = ' '.join(result)
    return text if text else 'Could not extract text from this image'

interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(type='pil', label='Upload Urdu Image'),
    outputs=gr.Textbox(label='Extracted Urdu Text'),
    title='Urdu OCR -- Code Saviours SI-26',
    description='Upload an image containing Urdu text and get the extracted text.',
    examples=[]
)
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b6cb51f3456c32998.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
